In [ ]:
import requests
import json
import pandas as pd
import os
import tqdm as tqdm
import requests
import pandas as pd
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.parse import quote_plus


# Define your API key
api_key = 'ntioungsta'

In [ ]:
# Load dataset
df = pd.read_csv('/Users/suhaibbasir/Documents/CS/MSc/Thesis/Thesis/EDP/europeana_datasets - europeana_datasets.csv')
df.head()

#print len of dataset
print(len(df))

In [ ]:
# Get first dataset name
dataset_name = df.iloc[0][0]
print(dataset_name)

In [ ]:
# Define the number of rows per request
rows = 100

# Initialize cursor for pagination
cursor = '*'

# Initialize list to store document IDs and documents
document_ids = []
documents = []

# Set a limit for the maximum number of requests
max_requests = 100  # Adjust based on how many documents you need
current_request = 0

# Function to get documents using cursor-based pagination
def get_documents(api_key, dataset_name, rows, cursor):
    search_url = f'https://api.europeana.eu/record/v2/search.json?wskey={api_key}&query=*&qf=edm_datasetName:"{dataset_name}"&rows={rows}&cursor={cursor}&profile=minimal&sort=random_1 asc, europeana_id asc'
    response = requests.get(search_url)
    data = response.json()
    print(data)
    # Check if 'items' key is in the response
    if 'items' in data:
        document_ids = [item['id'] for item in data['items']]
    else:
        document_ids = []
    
    next_cursor = data.get('nextCursor', None)
    safe_next_cursor = quote_plus(next_cursor) if next_cursor else None
    # print(f'Next cursor: {next_cursor}')
    return document_ids, safe_next_cursor

# Function to fetch document by ID
def fetch_document(api_key, doc_id):
    record_url = f'https://api.europeana.eu/record/v2/{doc_id}.rdf?wskey={api_key}'
    response = requests.get(record_url)
    return response.text

# Loop to paginate through results
while cursor and current_request < max_requests:
    print(f'Request #{current_request + 1}')
    current_request += 1
    new_document_ids, cursor = get_documents(api_key, dataset_name, rows, cursor)
    
    document_ids.extend(new_document_ids)

    # Debug information
    print(f'Cursor: {cursor}')
    print(f'Number of documents retrieved in this request: {len(new_document_ids)}')
    print(f'Total documents retrieved so far: {len(document_ids)}')

    # Fetch each document by its ID using concurrent requests
    with ThreadPoolExecutor(max_workers=10) as executor:
        future_to_doc = {executor.submit(fetch_document, api_key, doc_id): doc_id for doc_id in new_document_ids}
        for future in as_completed(future_to_doc):
            try:
                documents.append(future.result())
            except Exception as e:
                print(f"An error occurred: {e}")
    
    # To avoid hitting the API rate limit
    time.sleep(1)  # Adjust sleep time as needed

# Check the number of documents collected
print(f'Total documents collected: {len(documents)}')

In [ ]:
print(f'Total documents collected: {len(documents)}')

In [ ]:
def save_documents(dataset_name, documents):
    if not os.path.exists(dataset_name):
        os.makedirs(dataset_name)

    for i, doc in enumerate(documents):
        with open(f'{dataset_name}/document_{i}.rdf', 'w') as f:
            f.write(doc)

save_documents(dataset_name, documents)

In [ ]:
# Function to get document IDs from a dataset
def get_document_ids(dataset_name, rows):
    search_url = f'https://api.europeana.eu/record/v2/search.json?wskey={api_key}&query=*&qf=edm_datasetName:"{dataset_name}"&rows={rows}&sort=random_1%20asc&profile=minimal'
    response = requests.get(search_url)
    data = response.json()
    document_ids = [item['id'] for item in data['items']]
    return document_ids

# Function to download documents using document IDs
def download_documents(document_ids, format='rdf'):
    documents = []
    for doc_id in document_ids:
        record_url = f'https://api.europeana.eu/record/v2/{doc_id}.{format}?wskey={api_key}'
        response = requests.get(record_url)
        documents.append(response.text)
    return documents

In [ ]:
import os

def download_documents(document_ids, format='json'):
    documents = []
    folder_path = 'collection_experiment'
    os.makedirs(folder_path, exist_ok=True)  # Create the folder if it doesn't exist
    for doc_id in document_ids:
        record_url = f'https://api.europeana.eu/record/v2/{doc_id}.{format}?wskey={api_key}'
        response = requests.get(record_url)
        file_path = os.path.join(folder_path, f'{doc_id}.{format}')
        with open(file_path, 'w') as file:
            file.write(response.text)
        documents.append(file_path)
    return documents